# Step 1 — Hash functions (MD5 + ga4gh SQ)

Establish and verify the two digests before touching any real genomic data, per `HANDOFF.md`:

- **MD5** of normalized residues (matches the SAM/BAM `M5` tag, samtools-aligned).
- **ga4gh SQ** = `base64url( SHA-512(seq)[:24] )`, no padding -> `SQ.<32 chars>` (refget / VRS form: `ga4gh:SQ.<...>`).

The two digests are independent — SQ is not derived from MD5. Both are kept as columns in the eventual SQLite catalog.

**Digest decision:** MD5 is primary (samtools/refget-aligned); SQ is stored alongside as the stronger digest. No third digest (SHA-256) needed.

In [11]:
import hashlib
import base64
import re

## Normalization (protein)

Mandatory before hashing, per handoff:
- Uppercase; residues only; strip whitespace/newlines.
- Keep initiator Met.
- No trailing stop (`*`) — stop encodes no amino acid, and public protein hashes (UniProt/GENCODE/refget) exclude it.

CDS/nucleotide normalization (stop-codon inclusion, low-complexity flagging) is deferred to the notebook that handles CDS extraction — different rules apply there.

In [12]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYUOBZJX")  # includes selenocysteine (U), pyrrolysine (O), ambiguity codes

def normalize_protein(seq: str) -> str:
    seq = re.sub(r"\s+", "", seq).upper()
    seq = seq.rstrip("*")
    invalid = set(seq) - VALID_AA
    if invalid:
        raise ValueError(f"Unexpected residues in protein sequence: {invalid}")
    return seq

## Digest functions

In [13]:
def md5_digest(normalized_seq: str) -> str:
    return hashlib.md5(normalized_seq.encode("ascii")).hexdigest()

def ga4gh_sq_digest(normalized_seq: str) -> str:
    digest = hashlib.sha512(normalized_seq.encode("ascii")).digest()[:24]
    b64 = base64.urlsafe_b64encode(digest).decode("ascii").rstrip("=")
    return f"SQ.{b64}"

## Verify against the worked example

The handoff's worked example cites accession `NP_000207.1` for human preproinsulin (110 aa). That accession is **wrong** —
`NP_000207.1` is actually Kallmann syndrome 1 protein (ANOS1, 787 aa), confirmed live against NCBI eutils on 2026-07-02.

The correct accession for human preproinsulin is **`NP_000198.1`**, also confirmed live against NCBI eutils (110 aa,
starts `MALWMRLL...`, ends `...LYQLENYCN`). Its sequence reproduces the handoff's MD5/SQ values exactly, confirming
the digest *formulas* were right all along — only the accession label was mistyped from memory.

In [ ]:
# NP_000198.1 insulin preproprotein [Homo sapiens] -- fetched live from NCBI eutils 2026-07-02
preproinsulin = (
    "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGG"
    "GPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
)

normalized = normalize_protein(preproinsulin)
assert len(normalized) == 110, f"expected 110 aa, got {len(normalized)}"

md5 = md5_digest(normalized)
sq = ga4gh_sq_digest(normalized)

assert md5 == "12e9c9e4e2835c302e8ba615115edda3", md5
assert sq == "SQ.W3vopEox9qIpHK2i2i8f_YnHXK-GZOwv", sq

print("accession: NP_000198.1 (corrected from handoff's NP_000207.1)")
print("length:   ", len(normalized))
print("MD5:      ", md5)
print("SQ:       ", sq)
print("ga4gh:SQ. form:", f"ga4gh:{sq}")
print("All assertions passed.")

## Next steps

1. Extend `normalize_*` with a CDS/nucleotide variant (decide + document stop-codon inclusion).
2. Download GENCODE v46 chr22 subset (GTF + `pc_transcripts.fa` + `pc_translations.fa`) into `data/reference/` — release already pinned and MD5-verified in `data/reference/md5sum.txt`.
3. Parse GTF `CDS` features per transcript, splice into full CDS, translate, and cross-validate against `pc_translations.fa` (`translate(CDS) == protein` invariant from the handoff).
4. Emit per-exon CDS hashes alongside the whole-CDS and protein hashes.
5. Land everything in the SQLite catalog schema from the handoff.

Promote this notebook's functions into `scripts/` once the CDS-extraction notebook stabilizes them.